# §3.9 Validation Pipeline (Amendment 5)

Runs the Colab portion of the AutoPET-III §3.9 outlier-handling validation:
1. Draws a stratified random sample of 50 lesions
2. Renders per-lesion review pages (anonymised; only Review ID visible)

The manual review (~2.5 hours of reviewer work) and kappa analysis run on your Mac, AFTER this notebook completes both steps.

**Pre-registration:** https://doi.org/10.17605/OSF.IO/4KAZN (Amendment 5)  
**OSF amendment_log v6 SHA-256:** `9e692af0a849152c3d7b798921473f376e2029842101677bc05a9a2f77bcd26a`

## CRITICAL blinding rule

When you download review materials to your Mac after this notebook completes, download ONLY:
- `sample_blinded.csv` (50 rows with empty `reviewer_decision` column)
- `pngs/review_01.png` through `review_50.png` (50 anonymised lesion images)

Do **NOT** download or open `sample_key.csv` until your blinded review is complete. The key reveals case_id, triage_category, the SUVpeak/SUVmax ratio, and the index test's prediction — seeing any of these during review breaks the validation protocol.

## Runtime

- **Hardware:** CPU runtime sufficient (no GPU needed)
- **Wall-clock:** ~10 seconds for sampling, ~30-60 minutes for rendering
- **Drive output:** `$WORK_DIR/autopet_iii/section_3_9_validation/`

In [ ]:
# Working-directory configuration:
# Set the WORK_DIR environment variable (or override here) to point at
# the local or networked folder that holds the raw cohort data.
# Default Colab convention: the mounted Drive root.
import os
WORK_DIR = os.environ.get('WORK_DIR', '/content/drive/MyDrive')
print(f'WORK_DIR = {WORK_DIR}')


In [ ]:
# Step 1: Mount Drive + verify required inputs
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = f'{WORK_DIR}/autopet_iii'
OUT_DIR = f'{DRIVE}/section_3_9_validation'
PNG_DIR = f'{OUT_DIR}/pngs'

required = [
    f'{DRIVE}/autopet_iii_lesions.parquet',
    f'{DRIVE}/_pt_ct_pairs.csv',
    f'{DRIVE}/segmentations',
]
all_ok = True
for p in required:
    if os.path.exists(p):
        print(f'OK      {p}')
    else:
        print(f'MISSING {p}')
        all_ok = False
if not all_ok:
    raise RuntimeError('Required input(s) missing -- stop and investigate before proceeding')

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(PNG_DIR, exist_ok=True)
print(f'\nOutput directories ready:')
print(f'  CSVs: {OUT_DIR}')
print(f'  PNGs: {PNG_DIR}')

In [ ]:
# Step 2: Install Python dependencies
# pydicom + SimpleITK + nibabel + scipy + matplotlib are needed for the renderer.
# pandas + pyarrow are needed for the sampler. tcia_utils is not needed here.
!pip install -q pydicom SimpleITK nibabel scipy matplotlib pandas pyarrow

In [ ]:
# Step 3: Upload the three Python files needed for this pipeline.
#
# The three files live in different folders on your Mac:
#   - scripts/section_3_9_validation_sample.py
#   - scripts/section_3_9_validation_render.py
#   - src/preprocess/suv_conversion.py
#
# Colab's file picker only opens one folder at a time, so this cell prompts
# repeatedly until all three files are uploaded. On each picker:
#   1. Navigate to the folder containing one or more of the missing files
#   2. Select all matching files (you can shift/cmd-click within ONE folder)
#   3. Click Open. The cell will check what's still needed and prompt again.
from google.colab import files

needed = {
    'section_3_9_validation_sample.py',
    'section_3_9_validation_render.py',
    'suv_conversion.py',
}

# Clear any stale prior copies
for name in needed:
    p = f'/content/{name}'
    if os.path.exists(p):
        os.remove(p)

have = set()
round_num = 0
while have != needed:
    round_num += 1
    missing = needed - have
    print(f'\n--- Upload round {round_num}: still need {sorted(missing)} ---')
    uploaded = files.upload()
    new = set(uploaded) & needed
    if not new:
        print(f'(no relevant files in that batch -- ignoring)')
    have |= new

for name in needed:
    print(f'OK uploaded {name}  ({os.path.getsize("/content/" + name)} bytes)')

In [ ]:
# Step 4: Draw the stratified random sample
# Outputs sample_blinded.csv + sample_key.csv directly to Drive at OUT_DIR.
# Seed=42 (project default); deterministic given the parquet content.
import subprocess

result = subprocess.run([
    'python', '/content/section_3_9_validation_sample.py',
    '--parquet', f'{DRIVE}/autopet_iii_lesions.parquet',
    '--out',     OUT_DIR,
], capture_output=False)
if result.returncode != 0:
    raise RuntimeError(f'Sampling failed with exit code {result.returncode}')

# Verify the two CSVs landed
import os
for fname in ('sample_blinded.csv', 'sample_key.csv'):
    p = f'{OUT_DIR}/{fname}'
    assert os.path.exists(p), f'Expected {p} to exist after sampling'
    print(f'OK {p}  ({os.path.getsize(p)} bytes)')

In [ ]:
# Step 5: Quick visual check of the sample structure
# Confirms the 50-lesion stratified sample looks right before committing to render.
import pandas as pd
key = pd.read_csv(f'{OUT_DIR}/sample_key.csv')
print(f'Sample size: {len(key)}')
print(f'Per stratum: {key["triage_category"].value_counts().to_dict()}')
print(f'Unique cases: {key["case_id"].nunique()} (one-lesion-per-case cap honoured if equals sample size)')
print(f'Index predicts retain: {(key["index_predicted_decision"]=="retain").sum()}')
print(f'Index predicts exclude: {(key["index_predicted_decision"]=="exclude").sum()}')
print()
blinded = pd.read_csv(f'{OUT_DIR}/sample_blinded.csv')
print(f'Blinded CSV columns: {list(blinded.columns)}')
print(f'Reviewer decision column empty? {(blinded["reviewer_decision"].isna() | (blinded["reviewer_decision"]=="")).all()}')

In [ ]:
# Step 6: Render the 50 review pages
# Loads each PT series ONCE and renders all sampled lesions for that case before moving on.
# Per-case cost: ~30-60 seconds (DICOM extract + SUV compute + CT load + resample + N renders).
# 50 unique cases (one-lesion-per-case cap) -> wall-clock ~30-60 min total.
import subprocess

result = subprocess.run([
    'python', '/content/section_3_9_validation_render.py',
    '--key',        f'{OUT_DIR}/sample_key.csv',
    '--drive-root', DRIVE,
    '--suv-module', '/content/suv_conversion.py',
    '--out',        PNG_DIR,
], capture_output=False)
print(f'\nRender exit code: {result.returncode}')

In [ ]:
# Step 7: Verify all 50 PNGs landed on Drive
import os
pngs = sorted(f for f in os.listdir(PNG_DIR) if f.endswith('.png'))
print(f'PNGs on Drive: {len(pngs)} / 50 expected')
if len(pngs) > 0:
    print(f'First 3: {pngs[:3]}')
    print(f'Last 3:  {pngs[-3:]}')
    sizes = [os.path.getsize(os.path.join(PNG_DIR, f)) for f in pngs]
    print(f'Size range: {min(sizes)/1024:.0f}-{max(sizes)/1024:.0f} KB (median {sorted(sizes)[len(sizes)//2]/1024:.0f} KB)')

missing_ids = sorted(set(range(1, 51)) - {int(f[len('review_'):-len('.png')]) for f in pngs})
if missing_ids:
    print(f'\nMissing review_ids: {missing_ids}')
    print(f'Check the render output above for [FAIL] entries -- usually a missing CT match or a SEG shape mismatch.')
else:
    print(f'\nAll 50 PNGs present.')

## Step 8: Manual review (your action, on your Mac)

Colab portion is done. Now download the review materials to your Mac and complete the blinded review.

### What to download

From `$WORK_DIR/autopet_iii/section_3_9_validation/`:

- `sample_blinded.csv` (the spreadsheet for filling in decisions)
- All 50 PNGs in `pngs/` (download the folder as a zip via Drive web UI, or sync via Drive desktop)

**Do NOT download or open `sample_key.csv`.** That file is the unblinding key; it MUST stay sealed until your review is complete.

### Review protocol

1. Open `sample_blinded.csv` in Excel / Numbers / Google Sheets
2. For each review_id 1-50, in order:
   - Open `pngs/review_NN.png`
   - The image shows: 5 axial CT+SUV thumbnails, a coronal MIP, a sagittal MIP. Only the Review ID is visible.
   - Decide: `retain` (real biology) or `exclude` (artefact / contamination)
   - Type the decision in the `reviewer_decision` cell for that row
   - Optionally: `reviewer_confidence` (`high` / `medium` / `low`), `reviewer_notes` (free text)
3. Save the spreadsheet back as **CSV** (not .xlsx) at the same filename `sample_blinded.csv`

Pacing: ~3 minutes per lesion is realistic. Take breaks. Don't speed-read — fatigue affects review quality.

### Step 9: Run kappa analysis (your Mac, ~5 seconds)

Once all 50 decisions are recorded:

```bash
# Move filled-in CSV into the project results dir
mkdir -p "/Users/haydenfarquhar/Documents/Research Projects/79 Conformal SUV Theranostic/results/tables/section_3_9_validation"
cp ~/Downloads/sample_blinded.csv \
   "/Users/haydenfarquhar/Documents/Research Projects/79 Conformal SUV Theranostic/results/tables/section_3_9_validation/"

# Now download sample_key.csv from Drive (review is complete -- safe to unblind)
# Place it in the same folder.

# Run analysis
cd "/Users/haydenfarquhar/Documents/Research Projects/79 Conformal SUV Theranostic"
python3 scripts/section_3_9_validation_analyse.py
```

Output: `results/tables/section_3_9_validation/kappa_report.md` plus a console summary with the pre-specified action decision (one of `SUBSTANTIAL_AGREEMENT` / `MODERATE_AGREEMENT` / `INSUFFICIENT_AGREEMENT`).

Send me the kappa report's decision text and I'll guide the next step (apply ratio rule, refine threshold, or revert to full review).